In [ ]:
from datasets import load_dataset
import pandas as pd
from itertools import islice

In [ ]:
# Load the single Parquet file (non-streaming mode)
ds = load_dataset(
    "defeatbeta/yahoo-finance-data",
    data_files="data/stock_news.parquet",
    split="train"
)

# Convert to pandas DataFrame
df = ds.to_pandas()

# Preview first 5 rows
#print(df.head())

In [ ]:
df.tail()

,uuid,related_symbols,title,publisher,report_date,type,link,news
751343,a840a323-3ddd-3c18-b12b-284ff005dc0c,ZYME,How Recent Developments Are Changing The Zymew...,Simply Wall St.,2025-10-23,STORY,https://finance.yahoo.com/news/recent-developm...,"[{'paragraph_number': 1, 'highlight': '', 'par..."
751344,aa223c10-1c1a-3fd3-8bca-2e299f01cff2,ZYME,Zymeworks (ZYME): Assessing Valuation After 60...,Simply Wall St.,2025-11-19,STORY,https://finance.yahoo.com/news/zymeworks-zyme-...,"[{'paragraph_number': 1, 'highlight': '', 'par..."
751345,873163dc-2375-37be-ad7a-68d7863fd93d,ZYXIQ,New Zynex Leadership Responds to Indictment of...,PR Newswire,2026-01-23,STORY,https://finance.yahoo.com/news/zynex-leadershi...,"[{'paragraph_number': 1, 'highlight': '', 'par..."
751346,b02d458b-d8b2-353d-bdd8-a54c574ffd59,ZYXIQ,Zynex Poised for New Future with Resolution of...,PR Newswire,2026-02-18,STORY,https://finance.yahoo.com/news/zynex-poised-fu...,"[{'paragraph_number': 1, 'highlight': '', 'par..."
751347,0824094d-8ca4-32f1-a4fc-3c68021bf019,ZYXIQ,Zynex Announces New In-Network Provider Status...,PR Newswire,2026-02-04,STORY,https://finance.yahoo.com/news/zynex-announces...,"[{'paragraph_number': 1, 'highlight': '', 'par..."


In [ ]:
df['news'].iloc[0]


array([{'paragraph_number': 1, 'highlight': '', 'paragraph': 'UC Davis researcher recognized for pioneering analytical approaches in peptide drug development\nSANTA CLARA, Calif., November 12, 2025--(BUSINESS WIRE)--Agilent Technologies Inc. (NYSE: A) today announced that the University of California has received an Agilent Research Catalyst (ARC) award on behalf of Professor Marie Heffern at the UC Davis campus. The award recognizes Professor Heffern’s innovative research into the analytical characterization of peptide and protein therapeutics targeting incretin and related pathways, an emerging area critical to advancing treatments for diabetes and obesity.\nProfessor Heffern is a faculty member in the Department of Chemistry at UC Davis. Her research focuses on understanding how trace metal ions, such as copper, iron, and zinc, interact with incretin-based peptide therapeutics, including GLP-1 receptor agonists. These interactions play a critical role in drug stability, bioactivity,

In [ ]:
for item in df['news'].iloc[0]:
    print(item.keys())

dict_keys(['paragraph_number', 'highlight', 'paragraph'])


In [ ]:
df['related_symbols'].describe()

,related_symbols
count,751348
unique,8333
top,NVDA
freq,7204


In [ ]:
skip_cols = ['uuid', 'link', 'news']  # add 'news' or others if they hang too

for col in df.columns:
    if col not in skip_cols:
        print(f"\n=== {col} ===")
        print(df[col].describe())


=== related_symbols ===
count     751348
unique      8333
top         NVDA
freq        7204
Name: related_symbols, dtype: object

=== title ===
count                    751348
unique                   445943
top       BC-Most Active Stocks
freq                       1746
Name: title, dtype: object

=== publisher ===
count     751348
unique       291
top        Zacks
freq      141173
Name: publisher, dtype: object

=== report_date ===
count         751348
unique           370
top       2026-02-27
freq            5633
Name: report_date, dtype: object

=== type ===
count     751348
unique         2
top        STORY
freq      741891
Name: type, dtype: object


In [ ]:
# Peek at raw types and values
print(df['related_symbols'].apply(type).value_counts())  # should show <class 'str'> or <class 'list'>

# Sample values
print(df['related_symbols'].head(10))
print(df['related_symbols'].sample(10))  # random ones
print(df['related_symbols'].isnull().sum())  # how many null/NaN

related_symbols
<class 'str'>    751348
Name: count, dtype: int64
0    A
1    A
2    A
3    A
4    A
5    A
6    A
7    A
8    A
9    A
Name: related_symbols, dtype: object
307027      GPK
349217     INCY
252670      EXR
595299     SBGI
646384      TAL
476898    NOPFF
286378      GCO
511837     ORCL
336124     HUBB
544534     POOL
Name: related_symbols, dtype: object
0


In [ ]:
# Quick scan for suspicious values
multi_candidates = df[
    df['related_symbols'].str.contains(',', na=False) |
    df['related_symbols'].str.contains(' ', na=False) &
    ~df['related_symbols'].str.match(r'^[A-Z]{1,5}$', na=False)  # rough non-single-ticker filter
]
print(len(multi_candidates))
print(multi_candidates['related_symbols'].head(10) if not multi_candidates.empty else "None found")


0
None found


In [ ]:
print(df['related_symbols'].value_counts().head(10))  # e.g., NVDA might top with 7204

related_symbols
NVDA    7204
AMZN    3602
MSFT    3295
TSLA    2516
META    2440
AAPL    2429
INTC    2338
GOOG    2254
FDS     2023
PLTR    1782
Name: count, dtype: int64


In [ ]:
# Assuming you already have df from earlier:
# df = load_dataset(...).to_pandas()

# Create a new column with the count of tickers per row
df['num_symbols'] = df['related_symbols'].apply(
    lambda x: len(x) if isinstance(x, list) else 0
)

# Now get describe() on the counts
print(df['num_symbols'].describe())

# Or for a quick frequency table (how many rows have exactly N tickers)
print(df['num_symbols'].value_counts().sort_index().head(20))  # top 20 most common counts

count    751348.0
mean          0.0
std           0.0
min           0.0
25%           0.0
50%           0.0
75%           0.0
max           0.0
Name: num_symbols, dtype: float64
num_symbols
0    751348
Name: count, dtype: int64


In [ ]:
# Define the top 10 tickers once (same as in the prices notebook)
top10_tickers = ['NVDA', 'AMZN', 'MSFT', 'TSLA', 'META',
                 'AAPL', 'INTC', 'GOOG', 'FDS', 'PLTR']

# Filter the full news df (which is called 'df' in this notebook)
top10_news_df = df[df['related_symbols'].isin(top10_tickers)].copy()

# Sort for time-series friendliness
top10_news_df = top10_news_df.sort_values(['related_symbols', 'report_date'])

# Quick verification
print("Shape of top10_news_df:", top10_news_df.shape)
print("\nUnique tickers:", sorted(top10_news_df['related_symbols'].unique()))
print("\nDate range:", top10_news_df['report_date'].min(), "to", top10_news_df['report_date'].max())

Shape of top10_news_df: (29883, 8)

Unique tickers: ['AAPL', 'AMZN', 'FDS', 'GOOG', 'INTC', 'META', 'MSFT', 'NVDA', 'PLTR', 'TSLA']

Date range: 2025-03-12 to 2026-03-15


In [ ]:
print(df['report_date'].min())
print(df['report_date'].max())



2025-03-11
2026-03-15


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
%cd "drive/MyDrive/Colab Notebooks/MIS 769/AnalyzeMe2"

/content/drive/MyDrive/Colab Notebooks/MIS 769/AnalyzeMe2


In [ ]:
# Save the filtered top-10 news DataFrame
top10_news_df.to_parquet("top10_stock_news.parquet", index=False)
print("Saved to top10_stock_news.parquet")

Saved to top10_stock_news.parquet
